# Baselines

Two reference "models" that every real model has to beat:

1. **Constant**: always predicts the mean `cnt` of the training part.
2. **Hour x day type**: predicts the mean `cnt` of the training part for the
   given pair (`hr`, `workingday`).

Both are fitted **inside each fold** of the validation scheme (via
`evaluate`), so the means never include validation data (no leakage).

In [ ]:
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.dummy import DummyRegressor

from src.data import load_data
from src.validation import SEED, evaluate

X, y = load_data()
print(X.shape, y.shape)

## Hour x day type baseline

A tiny sklearn-compatible estimator. It inherits from `BaseEstimator`, so
`clone` (used in `evaluate`) works. Pairs that were not seen in the training
part fall back to the training mean.

In [ ]:
class HourDayBaseline(RegressorMixin, BaseEstimator):
    """Mean of the target for each (hr, workingday) pair of the training part."""

    def fit(self, X, y):
        d = X[["hr", "workingday"]].copy()
        d["y"] = y.to_numpy()
        self.table_ = d.groupby(["hr", "workingday"])["y"].mean()
        self.global_mean_ = float(d["y"].mean())
        return self

    def predict(self, X):
        keys = pd.MultiIndex.from_frame(X[["hr", "workingday"]])
        return self.table_.reindex(keys).fillna(self.global_mean_).to_numpy()

## Evaluation in both schemes

In [ ]:
baselines = {
    "baseline_const": DummyRegressor(strategy="mean"),
    "baseline_hour_day": HourDayBaseline(),
}

summary = []
for name, model in baselines.items():
    for scheme in ["random", "chrono"]:
        folds, preds = evaluate(model, X, y, scheme, name)
        summary.append({
            "model": name,
            "scheme": scheme,
            "rmse_mean": folds["rmse"].mean(),
            "rmse_std": folds["rmse"].std(),
            "mae_mean": folds["mae"].mean(),
            "r2_mean": folds["r2"].mean(),
        })

pd.DataFrame(summary).round(3)

## Per-fold results (chronological scheme)

The folds cover different periods, so their errors are not interchangeable.

In [ ]:
pd.read_csv("results/baseline_hour_day_chrono_folds.csv")

## Conclusions

_To be written after looking at the numbers._